# 全量微调对比实验（A100）

**第三组对比**：不使用 LoRA/DoRA，直接全量微调 Qwen2.5-1.5B 的全部参数。

| 对比组 | 方法 | 可训练参数 |
|--------|------|------------|
| 组1 | LoRA (r=16) | ~4M |
| 组2 | DoRA (r=16) | ~4M |
| **组3** | **全量微调** | **~1500M** |

1.5B bf16 ≈ 3GB，A100-80GB 可以直接全量微调，不需要量化。

In [ ]:
# Cell 1：挂载 Drive + Token
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!mkdir -p "{DRIVE_DIR}/outputs"
print("✅ Drive 已挂载，HF Token 已加载")

In [ ]:
# Cell 2：克隆项目 + 安装依赖
import os

PROJECT_DIR = "/content/6000Q-QwenMiniReason"
REPO_URL = f"https://{os.environ['HF_TOKEN']}@github.com/yukiiii0730/6000Q-QwenMiniReason.git"
BRANCH = "Nancy"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} reset --hard origin/{BRANCH}

%cd {PROJECT_DIR}
!pip install -r requirements.txt -q
print("✅ 项目已就绪")

In [ ]:
# Cell 3：生成全量微调专用配置
import yaml

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

# ── 模式切换 ───────────────────────────────────────────────────
FAST_TEST = True   # True=快速测试, False=正式训练

if FAST_TEST:
    MAX_STEPS = 50
    print("⚡ 快速测试模式")
else:
    MAX_STEPS = 1800
    print("🔥 正式训练模式")

# 读取基础配置，覆盖为全量微调参数
with open("config/sft_config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

cfg["output_dir"] = f"{DRIVE_DIR}/outputs/sft_full"   # 隔离输出
cfg["load_in_4bit"] = False                            # 不量化
cfg.pop("datasets", None)
cfg.pop("lora", None)                                  # 不需要 LoRA 配置

# 全量微调超参调整：
#   - lr 降到 2e-5（全量微调不需要 LoRA 那么大的学习率）
#   - batch_size=4, ga=4（全量微调显存占用更大）
#   - optim 用标准 adamw（不用量化优化器）
cfg["train"]["learning_rate"] = 2e-5
cfg["train"]["per_device_train_batch_size"] = 4
cfg["train"]["gradient_accumulation_steps"] = 4
cfg["train"]["max_steps"] = MAX_STEPS
cfg["train"]["optim"] = "adamw_torch"
cfg["train"]["bf16"] = True
cfg["train"]["fp16"] = False

with open("config/sft_config.yaml", "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print(f"📋 全量微调配置：")
print(f"   输出: {cfg['output_dir']}")
print(f"   lr={cfg['train']['learning_rate']}, bs=4×4=16, steps={MAX_STEPS}")
print(f"   量化=否, LoRA=否")
print("✅ 配置已生成")

In [ ]:
# Cell 4：准备数据（复用缓存）
import json, os
from datasets import load_dataset

DRIVE_DIR  = "/content/drive/MyDrive/Qwen-Reasoning"
PROCESSED  = f"{DRIVE_DIR}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

SFT_OUT = f"{PROCESSED}/sft_train.json"
DEFAULT_INSTRUCTION = "请先进行清晰的逐步推理，再给出最终答案。"

if FAST_TEST:
    SFT_NUMINA_N, SFT_MAGPIE_N = 500, 500
else:
    SFT_NUMINA_N, SFT_MAGPIE_N = 50000, 50000

if os.path.exists(SFT_OUT):
    print(f"⏭️  SFT 缓存已存在: {SFT_OUT}")
else:
    print("🔄 转换 SFT 数据集...")
    sft_rows = []
    numina = load_dataset("AI-MO/NuminaMath-CoT", split="train", trust_remote_code=True).select(range(SFT_NUMINA_N))
    for x in numina:
        sft_rows.append({"instruction": DEFAULT_INSTRUCTION, "input": x["problem"].strip(), "output": x["solution"].strip()})
    magpie = load_dataset("Magpie-Align/Magpie-Reasoning-150K", split="train", trust_remote_code=True).select(range(SFT_MAGPIE_N))
    for x in magpie:
        sft_rows.append({"instruction": x["instruction"].strip(), "input": "", "output": x["response"].strip()})
    with open(SFT_OUT, "w", encoding="utf-8") as f:
        json.dump(sft_rows, f, ensure_ascii=False, indent=2)
    print(f"✅ SFT: {len(sft_rows)} 条")

# 写回 config
import yaml
with open("config/sft_config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
cfg["dataset_path"] = SFT_OUT
with open("config/sft_config.yaml", "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print("✅ 数据准备完成")

In [ ]:
# Cell 5：全量微调训练
!python scripts/full_finetune.py --config config/sft_config.yaml

In [ ]:
# Cell 6：GSM8K 评测
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
GSM8K_N = 50 if FAST_TEST else 1319

# 全量微调的模型直接保存在输出目录（不需要 merge_lora）
!python eval/gsm8k_eval.py \
    --model_path "{DRIVE_DIR}/outputs/sft_full" \
    --max_samples {GSM8K_N} \
    --output eval/gsm8k_full_ft_result.json

import json
with open("eval/gsm8k_full_ft_result.json") as f:
    result = json.load(f)
print(f"\n✅ 全量微调 GSM8K: {result['accuracy']:.2%}  ({result['correct']}/{result['total']})")

In [ ]:
# Cell 7：三组对比（Baseline / LoRA / DoRA / 全量微调）
import json

files = {
    "Baseline (原始模型)": "eval/gsm8k_baseline.json",
    "LoRA (SFT+DPO)":     "eval/gsm8k_result.json",
    "DoRA (SFT+DPO)":     "eval/gsm8k_dora_result.json",
    "全量微调 (SFT only)": "eval/gsm8k_full_ft_result.json",
}

results = {}
for label, path in files.items():
    try:
        with open(path) as f:
            results[label] = json.load(f)["accuracy"]
    except FileNotFoundError:
        results[label] = None

baseline_acc = results.get("Baseline (原始模型)")

print("\n" + "=" * 55)
print(f"  {'模型':<25} {'GSM8K':>10} {'vs Baseline':>12}")
print("=" * 55)
for label, acc in results.items():
    acc_str = f"{acc:.2%}" if acc is not None else "N/A"
    delta = ""
    if acc is not None and baseline_acc is not None and label != "Baseline (原始模型)":
        delta = f"{acc - baseline_acc:+.2%}"
    print(f"  {label:<25} {acc_str:>10} {delta:>12}")
print("=" * 55)

# 找出最优
valid = {k: v for k, v in results.items() if v is not None and k != "Baseline (原始模型)"}
if valid:
    best = max(valid, key=valid.get)
    print(f"\n🏆 最优方法: {best} ({valid[best]:.2%})")

# 保存
with open("eval/three_way_compare.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"📝 对比结果: eval/three_way_compare.json")

In [ ]:
# Cell 8（可选）：推送结果到 GitHub
!git add eval/gsm8k_full_ft_result.json eval/three_way_compare.json
!git commit -m "Add full fine-tuning results & three-way comparison" || true
!git push origin Nancy || true